# DDInter → PharmPilot Interaction Bundle
Builds a `bundle-v1` SQLite from DDInter 2.0 and downloads it. Then install it in **Admin → Interaction Bundle** (#3c).

In [ ]:
# 1. Dependencies + PharmPilot package (for the SAME normalize() + schema constant).
!pip -q install pandas requests
# Option A: clone the repo so the builder is importable.
# !git clone <YOUR_REPO_URL> pharmpilot && pip -q install -e pharmpilot
# Option B: upload normalizer.py + bundle.py and adjust sys.path.


In [ ]:
# 2. Download DDInter 2.0 CSV exports (per-ATC). Update URLs to the current release.
import pandas as pd, requests, io
DDINTER_CSV_URLS = [
    # 'https://ddinter.scbdd.com/static/media/download/ddinter_downloads_code_A.csv',
    # ... add the per-category files you need ...
]
frames = []
for url in DDINTER_CSV_URLS:
    frames.append(pd.read_csv(io.StringIO(requests.get(url, timeout=60).text)))
raw = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
raw.head()


In [ ]:
# 3. Adapt DDInter columns -> RawInteraction rows.
# DDInter columns are typically: Drug_A, Drug_B, Level (Major/Moderate/Minor).
from scripts.interaction_bundle.ddinter_builder import RawInteraction
def to_rows(df):
    for _, r in df.iterrows():
        yield RawInteraction(str(r.get('Drug_A','')), str(r.get('Drug_B','')),
                             str(r.get('Level','')), str(r.get('Mechanism','') or ''))
rows = list(to_rows(raw))
len(rows)


In [ ]:
# 4. Build + write the bundle.
from scripts.interaction_bundle.ddinter_builder import build_rules, write_bundle
rules = build_rules(rows)
meta = write_bundle(rules, 'bundle.sqlite', {'ddinter': '2.0'})
print(meta)


In [ ]:
# 5. Download the bundle, then install it in Admin -> Interaction Bundle.
from google.colab import files  # Colab only
files.download('bundle.sqlite')


**Next:** in PharmPilot, open *Dashboards -> Interaction Bundle* and upload `bundle.sqlite`. It is validated + atomically installed + hot-reloaded; curated rules always win.